# Test notebook

In [2]:
print(1)

1


In [1]:
import json
import os
import re
import requests
import codecs
from datetime import datetime, timedelta
from bs4 import BeautifulSoup

In [8]:
BASE_URL = "https://understat.com"

ds = '2026-03-21'
print(f"🔍 Searching for matches on: {ds}")

# 1. Fetch the League Page with a professional User-Agent
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'}
response = requests.get(BASE_URL, headers=headers, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.content, 'html.parser')
scripts = soup.find_all('script')

🔍 Searching for matches on: 2026-03-21


In [14]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <base href="https://understat.com/"/>
  <title>
   xG stats for teams and players from the TOP European leagues
  </title>
  <meta charset="utf-8"/>
  <meta content="On this site, you will find the detailed Expected Goals statistic for teams and players from the English Premier League, La Liga, Bundesliga, Serie A and etc." name="description"/>
  <meta content="xG, expected goals, advanced football stats, epl, premier league, serie a, la liga, bundesliga" name="Keywords"/>
  <meta content="width=device-width, maximum-scale=1, initial-scale=1" name="viewport" user-scalable="no"/>
  <script async="" src="https://sdk.adlook.tech/inventory/core.js" type="text/javascript">
  </script>
  <link href="apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180"/>
  <link href="favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/>
  <link href="favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/>
  <link href="manifest.json" rel="man

In [9]:
scripts

[<script async="" src="https://sdk.adlook.tech/inventory/core.js" type="text/javascript"></script>,
 <script>
 			var THEME = localStorage.getItem("theme") || 'DARK';
 			document.body.className = "theme-" + THEME.toLowerCase();
 		</script>,
 <script src="//ajax.googleapis.com/ajax/libs/webfont/1.6.26/webfont.js" type="text/javascript"></script>,
 <script>
 		/*
 		var flagFontsLoading = false;
 		WebFont.load({
 			google: {
 				families: ['Barlow:500', 'Anton']
 			},
 			active: function(){
 				flagFontsLoading = true;
 			},
 			inactive: function(){
 				flagFontsLoading = true;
 			}
 		});
 		*/
 		var flagFontsLoading = true;
 		var BASE_URL = 'https://understat.com/',
 			PROMOTION =  JSON.parse('\x7B\x22id\x22\x3A\x227\x22,\x22name\x22\x3A\x22adsense\x22,\x22template\x22\x3A\x22\x3Cins\x20class\x3D\x5C\x22adsbygoogle\x5C\x22\x20style\x3D\x5C\x22display\x3Ablock\x5C\x22\x20data\x2Dad\x2Dclient\x3D\x5C\x22ca\x2Dpub\x2D7470116180195095\x5C\x22\x20data\x2Dad\x2Dslot\x3D\x5C\x2

In [3]:

raw_blob = None
# 2. Extract the JSON blob using Raw String Regex
# We look for the content inside JSON.parse('...') or JSON.parse("...")
for s in scripts:
    content = s.get_text()
    if 'JSON.parse' in content:
        # r"..." tells Python NOT to interpret backslashes in the regex itself
        match = re.search(r"JSON\.parse\s*\(['\"](.*?)['\"]\)", content)
        if match:
            raw_blob = match.group(1)
            break

if not raw_blob:
    raise ValueError(f"CRITICAL: No data blob found in HTML for {ds}. First 200 chars: {response.text[:200]}")


In [4]:

# 3. THE SENIOR DECODE: Convert hex (\xHH) to valid JSON characters
try:
    # We decode the raw escape sequences into a clean UTF-8 string
    decoded_string = codecs.decode(raw_blob, 'unicode_escape')
    print(f"decoded_string snippet: {repr(decoded_string[:10000])}")  # Show a snippet of the decoded string for debugging
    data = json.loads(decoded_string)
    print(f"data snippet: {json.dumps(data)[:10000]}")
    
    # Normalize: Handle single object (match_info) vs list (datesData)
    if isinstance(data, dict):
        data = [data]
        
except Exception as e:
    print(f"❌ Decode/JSON Error: {e}")
    print(f"Snippet of problematic blob: {repr(raw_blob[:100])}")
    raise


decoded_string snippet: '{"id":"7","name":"adsense","template":"<ins class=\\"adsbygoogle\\" style=\\"display:block\\" data-ad-client=\\"ca-pub-7470116180195095\\" data-ad-slot=\\"2199699885\\" data-ad-format=\\"auto\\" data-full-width-responsive=\\"true\\"><\\/ins>","js":"(adsbygoogle = window.adsbygoogle || []).push({});","countries_list_type":"black","clicks":"5652","is_user_click":false,"country":{"name":"United Kingdom","code":"GB"}}'
data snippet: {"id": "7", "name": "adsense", "template": "<ins class=\"adsbygoogle\" style=\"display:block\" data-ad-client=\"ca-pub-7470116180195095\" data-ad-slot=\"2199699885\" data-ad-format=\"auto\" data-full-width-responsive=\"true\"></ins>", "js": "(adsbygoogle = window.adsbygoogle || []).push({});", "countries_list_type": "black", "clicks": "5652", "is_user_click": false, "country": {"name": "United Kingdom", "code": "GB"}}


In [6]:

# 4. Filter for Match IDs on this specific logical date ('ds')
# Handles both 'date' and 'datetime' keys for future-proofing
match_ids = [
    m['id'] for m in data 
    if ds in str(m.get('date', m.get('datetime', '')))
]

if not match_ids:
    print(f"⚽ No matches scheduled for {ds}. Task complete.")
    print(f"NO_MATCHES_ON_{ds}")


⚽ No matches scheduled for 2026-03-21. Task complete.
NO_MATCHES_ON_2026-03-21


In [ ]:

# # 5. Scrape and Save Shot Data for each Match
# for m_id in match_ids:
#     print(f"📦 Scraping Match ID: {m_id}")
#     m_res = requests.get(f"https://understat.com{m_id}", headers=headers, timeout=15)
#     m_res.raise_for_status()
    
#     m_soup = BeautifulSoup(m_res.content, 'html.parser')
#     m_scripts = m_soup.find_all('script')
    
#     shots_json = None
#     for ms in m_scripts:
#         m_content = ms.get_text()
#         if 'var shotsData' in m_content:
#             # Same robust regex pattern for individual match pages
#             m_match = re.search(r"var shotsData\s*=\s*JSON\.parse\s*\(['\"](.*?)['\"]\)", m_content)
#             if m_match:
#                 shots_json = codecs.decode(m_match.group(1), 'unicode_escape')
#                 break
    
#     if shots_json:
#         # Save to the mounted MinIO/S3 volume
#         os.makedirs(RAW_STORAGE_PATH, exist_ok=True)
#         file_path = f"{RAW_STORAGE_PATH}/match_{m_id}_{ds}.json"
#         with open(file_path, "w") as f:
#             f.write(shots_json)
#         print(f"✅ Saved Bronze JSON: {file_path}")